Input : Silver/operated   
Output : Gold/flight_facts

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

StatementMeta(sparkpool1, 23, 6, Finished, Available, Finished, False)

Read Silver Layer

In [6]:
SILVER_PATH = "abfss://silver@flightdatalakegen2.dfs.core.windows.net/operated/"
GOLD_PATH = "abfss://gold@flightdatalakegen2.dfs.core.windows.net/flight_facts/"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(SILVER_PATH)

df = df.withColumn(
    "FL_DATE",
    F.to_date("FL_DATE")
)

StatementMeta(sparkpool1, 23, 7, Finished, Available, Finished, False)

Time Features

In [7]:
df = (df
.withColumn("YEAR", F.year("FL_DATE"))
.withColumn("MONTH", F.month("FL_DATE"))
.withColumn("DAY", F.dayofmonth("FL_DATE"))
.withColumn("DAY_OF_WEEK", F.dayofweek("FL_DATE"))
)

StatementMeta(sparkpool1, 23, 8, Finished, Available, Finished, False)

Departure Time Features

In [8]:
df = df.withColumn(
"DEP_HOUR",
(F.col("CRS_DEP_TIME") / 100).cast(IntegerType())
)

df = df.withColumn(
"DEP_TIME_BUCKET",
F.when((F.col("DEP_HOUR") >= 6) & (F.col("DEP_HOUR") < 12), "Morning")
.when((F.col("DEP_HOUR") >= 12) & (F.col("DEP_HOUR") < 18), "Afternoon")
.when((F.col("DEP_HOUR") >= 18) & (F.col("DEP_HOUR") < 24), "Evening")
.otherwise("Night")
)

StatementMeta(sparkpool1, 23, 9, Finished, Available, Finished, False)

Delay Classification

In [9]:
df = df.withColumn(
"IS_DELAYED",
F.when(F.col("ARR_DELAY") >= 15, 1).otherwise(0)
)

df = df.withColumn(
"OTP_FLAG",
F.when(F.col("ARR_DELAY") <= 15, 1).otherwise(0)
)

StatementMeta(sparkpool1, 23, 10, Finished, Available, Finished, False)

Delay Severity

In [10]:
df = df.withColumn(
"DELAY_SEVERITY",
F.when(F.col("ARR_DELAY") < 0, "Early")
.when(F.col("ARR_DELAY") < 15, "On Time")
.when(F.col("ARR_DELAY") < 60, "Minor")
.when(F.col("ARR_DELAY") < 180, "Moderate")
.otherwise("Severe")
)

StatementMeta(sparkpool1, 23, 11, Finished, Available, Finished, False)

Route and Distance

In [11]:
df = df.withColumn(
"ROUTE",
F.concat_ws("-", "ORIGIN", "DEST")
)

df = df.withColumn(
"HAUL_TYPE",
F.when(F.col("DISTANCE") < 500, "Short")
.when(F.col("DISTANCE") < 1500, "Medium")
.otherwise("Long")
)

StatementMeta(sparkpool1, 23, 12, Finished, Available, Finished, False)

Delay Cause

In [12]:
# -------------------------------------------------------------------
# Identify dominant delay cause
# -------------------------------------------------------------------

DELAY_COLS = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

# Replace nulls with 0 during calculation
df = df.withColumn(
    "MAX_DELAY",
    F.greatest(
        *[F.coalesce(F.col(c), F.lit(0)) for c in DELAY_COLS]
    )
)

df = df.withColumn(
    "DELAY_CAUSE",
    F.when(F.col("IS_DELAYED") == 0, "No Delay")
     .when(
         F.coalesce(F.col("CARRIER_DELAY"), F.lit(0)) == F.col("MAX_DELAY"),
         "Carrier"
     )
     .when(
         F.coalesce(F.col("WEATHER_DELAY"), F.lit(0)) == F.col("MAX_DELAY"),
         "Weather"
     )
     .when(
         F.coalesce(F.col("NAS_DELAY"), F.lit(0)) == F.col("MAX_DELAY"),
         "NAS"
     )
     .when(
         F.coalesce(F.col("SECURITY_DELAY"), F.lit(0)) == F.col("MAX_DELAY"),
         "Security"
     )
     .when(
         F.coalesce(F.col("LATE_AIRCRAFT_DELAY"), F.lit(0)) == F.col("MAX_DELAY"),
         "Late Aircraft"
     )
     .otherwise("Unknown")
)

StatementMeta(sparkpool1, 23, 13, Finished, Available, Finished, False)

Total Cause Delay

In [13]:
df = df.withColumn(
"TOTAL_CAUSE_DELAY",
F.coalesce(F.col("CARRIER_DELAY"), F.lit(0)) +
F.coalesce(F.col("WEATHER_DELAY"), F.lit(0)) +
F.coalesce(F.col("NAS_DELAY"), F.lit(0)) +
F.coalesce(F.col("SECURITY_DELAY"), F.lit(0)) +
F.coalesce(F.col("LATE_AIRCRAFT_DELAY"), F.lit(0))
)

StatementMeta(sparkpool1, 23, 14, Finished, Available, Finished, False)

Save Gold Layer

In [14]:
df.write \
.mode("overwrite") \
.partitionBy("YEAR") \
.parquet(GOLD_PATH)

print("Gold Layer created successfully.")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

StatementMeta(sparkpool1, 23, 15, Finished, Available, Finished, False)

Gold Layer created successfully.
Rows: 59,556,533
Columns: 41
